# Week 3, day 2 — Worksheet 05 SOLUTIONS: pivot and pivot_table   (L04)

Executed in the lab image (pandas 3.0.5) against the real
`data/orders_long.csv`. Every quoted number is what it actually printed.

Question 4 is the one to re-read. The deck's own worked example shows a total,
and the code as printed produces an average.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 05 — pivot and pivot_table. Run this once.
import pandas as pd

orders = pd.read_csv("data/orders_long.csv")

# One row per region-year: no repeated pairs, so pivot() will work.
unique_pairs = orders.groupby(["Region", "Year"])["Sales"].sum().round(2).reset_index()

print("raw orders:", orders.shape)
print("one row per region-year:", unique_pairs.shape)
print()
print(unique_pairs.head())

PART A — pivot: rearranging only

### Question 1

An `(8, 4)` table: regions down, years across.

`pivot()` took three column names and rearranged the rows into a grid. No
aggregation happened — it worked only because the input already had exactly
one row per region-year pair.

In [ ]:
p = unique_pairs.pivot(index="Region", columns="Year", values="Sales")
print(p.to_string())
print()
print("shape:", p.shape)

### Question 2

Pivoted total, `unique_pairs` total and raw `orders` total all agree at `1605576.22` to 2dp.

`pivot()` moved values without touching them, so the total is preserved.
That is the guarantee it gives and the reason to prefer it when it applies:
if the numbers change, something else went wrong.

In [ ]:
p = unique_pairs.pivot(index="Region", columns="Year", values="Sales")
print("pivoted total:      ", round(p.sum().sum(), 2))
print("unique_pairs total: ", round(unique_pairs["Sales"].sum(), 2))
print("raw orders total:   ", round(orders["Sales"].sum(), 2))

### Question 3

**raises** `ValueError: Index contains duplicate entries, cannot reshape`. -> Ontario 2012 alone has **87** rows.

The deck prints this as `# ValueError: duplicate entries cannot reshape`.
The real message leads with `Index contains` — close enough to recognise,
different enough to fail a literal search.

The cause is the grain question from the week 3, day 1 class: one row of `orders`
is one order, and 87 of them are Ontario-in-2012. `pivot()` has one cell to
put them in and no rule for choosing, so it refuses rather than picking one.

Refusing is the right behaviour. The alternative — silently keeping the
first or last — is how a report ends up showing one order's value as a
region's year.

In [ ]:
try:
    orders.pivot(index="Region", columns="Year", values="Sales")
except Exception as exc:
    print("%s: %s" % (type(exc).__name__, exc))

print()
print("rows for Ontario in 2012:",
      len(orders[(orders["Region"] == "Ontario") & (orders["Year"] == 2012)]))

PART B — pivot_table: rearranging and aggregating

### Question 4

Ontario/2012 cell -> **`1719.29`**. The sum of those orders is **`149578.59`**; their mean is `1719.29`, over `87` rows.

**The default `aggfunc` is `mean`.** The cell is the average order value,
not the total.

The deck is not wrong about this — its 'Anatomy' slide says so explicitly.
But its *worked example* two slides earlier shows a table of 140 and 110,
which are sums, next to code that would produce averages unless you add
`aggfunc='sum'`. Copy that slide and you get a table that looks like a sales
report and is an average-order-size report.

The two differ by a factor of 87 here. Nothing about the output says which
one you are looking at — the cells are numbers with no units and no header
saying 'mean'. Always pass `aggfunc` explicitly, even when the default is
what you want, so the next reader does not have to know the default.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year", values="Sales")
print(pt.round(2).to_string())
print()
cell = pt.loc["Ontario", 2012]
sub = orders[(orders["Region"] == "Ontario") & (orders["Year"] == 2012)]["Sales"]
print("pivot_table cell:", round(cell, 2))
print("sum of those orders:", round(sub.sum(), 2))
print("mean of those orders:", round(sub.mean(), 2))
print("rows behind the cell:", len(sub))

### Question 5

With `aggfunc="sum"` the grand total is `1605576.22`, matching `orders["Sales"].sum()`.

Now the cells are totals and the table reconciles with the source. That
reconciliation is the check worth running on any summary table you publish:
if the cells are sums, they must add up to the total you started with.

It does not work for means, counts or maxima — which is itself informative.
Only an additive aggregate can be checked this way.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year", values="Sales", aggfunc="sum")
print(pt.round(2).to_string())
print()
print("grand total:", round(pt.sum().sum(), 2))
print("raw total:  ", round(orders["Sales"].sum(), 2))

### Question 6

`aggfunc=["sum", "mean", "count"]` -> `(8, 12)` with a **2-level** column index like `('sum', 2009)`.

Three aggregates x four years = twelve columns, addressed by tuples. Ontario
2012 reads `sum 149578.59`, `mean 1719.29`, `count 87` — the three numbers
from Q4 in one row.

That is the most honest form of a summary table, because the count sits
next to the average and a reader can see how many rows the average is built
from. It is also twelve columns wide, which is why people leave the count
out and why averages over nine orders end up presented like averages over
three hundred.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year", values="Sales",
                        aggfunc=["sum", "mean", "count"])
print("shape:", pt.shape, "| column levels:", pt.columns.nlevels)
print("first 4 columns:", list(pt.columns[:4]))
print()
print(pt.loc["Ontario"].round(2).to_string())

### Question 7

`margins=True` adds an `All` row and column; the corner cell is `1605576.22`, matching the raw total.

Convenient, and correct here because the aggregate is a **sum**. Adding
row totals to a table of sums gives the right answer whichever way you add
them up.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year", values="Sales",
                        aggfunc="sum", margins=True)
print(pt.round(2).to_string())
print()
print("corner cell:", round(pt.loc["All", "All"], 2))
print("raw total:  ", round(orders["Sales"].sum(), 2))

### Question 8

The eight 2012 cells average **`1995.2`**, but the `All` margin reads **`1522.22`** — which is the mean of the `266` raw 2012 orders.

A 31% gap between two numbers in the same column, and the smaller one is
correct.

The margin is computed from the **underlying rows**, not from the cells
above it. Averaging the eight regional averages weights Nunavut's 9 orders
equally with Ontario's 302 — Nunavut's unusually high `5183.80` drags the
naive figure up. The margin weights every order once, which is what 'the
average 2012 order' means.

This is the mean-of-means trap, and `margins=True` is quietly doing the
right thing while looking like it is doing the naive thing. Which is worse
than either: a reader who checks the column by hand will get `1995.2` and
conclude the margin is broken.

Averages of averages are almost never what you want. If you must combine
group means, weight them by group size.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year", values="Sales",
                        aggfunc="mean", margins=True)
col = pt[2012].drop("All")
print("2012 cells:", col.round(2).tolist())
print()
print("mean of those 8 cells:", round(col.mean(), 2))
print("the 'All' margin:     ", round(pt.loc["All", 2012], 2))
print()
sub = orders[orders["Year"] == 2012]["Sales"]
print("mean of the raw 2012 orders:", round(sub.mean(), 2), "over", len(sub), "rows")

### Question 9

Technology pivots to `(8, 4)` with **2** `NaN` cells; `fill_value=0` gives **0**. -> the grand total is `819642.22` either way.

Same two Nunavut gaps as worksheets 01, 03 and 04, reached by a fourth
route. `pivot_table` has `fill_value` for the same reason `unstack` does,
and it carries the same warning: zero is a claim.

The grand total is unchanged because summing skips `NaN` and adding zero
does nothing — but as worksheet 03 Q7 showed, any *mean* over these cells
would move.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
a = tech.pivot_table(index="Region", columns="Year", values="Sales", aggfunc="sum")
b = tech.pivot_table(index="Region", columns="Year", values="Sales",
                     aggfunc="sum", fill_value=0)
print(a.round(2).to_string())
print()
print("NaN cells without fill_value:", int(a.isna().sum().sum()))
print("NaN cells with fill_value=0: ", int(b.isna().sum().sum()))
print()
print("grand total either way:", round(a.sum().sum(), 2), round(b.sum().sum(), 2))

### Question 10

`columns="Quarter"` -> **raises** `KeyError: 'Quarter'`.

There is no `Quarter` column — the file has `Year` and `Month`, and a
quarter would have to be derived.

A clean, immediate failure, and the last of the sheet's three ways to be
stopped: `pivot()` refusing duplicates (Q3), `pivot_table` refusing a
missing column (Q10), and — between them — `pivot_table` cheerfully
returning averages when you meant totals (Q4). Only two of those three stop
you.

In [ ]:
print("columns available:", list(orders.columns))
print(orders.pivot_table(index="Region", columns="Quarter",
                         values="Sales", aggfunc="sum"))